# Devoir NLP — Pipeline appliqué à un document PDF en arabe

**Étudiante :** INSAF EL KORACHI  
**Module :** Natural Language Processing (NLP)  
**Sujet :** Application d'un pipeline NLP sur un texte arabe extrait depuis un fichier PDF

## 1. Importation des bibliothèques

In [7]:
!pip install pdfplumber pymupdf PyPDF2 -q


In [8]:
import re
import pdfplumber
import fitz  # PyMuPDF
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer


## 2. Collecte des données : extraction du texte depuis le PDF

In [9]:
# Remplace ce chemin par le chemin réel de ton fichier PDF sur Kaggle
pdf_path = "/kaggle/input/datasets/elkorachiinsaf/arabic-1/arabic (1).pdf"


In [10]:
def extract_with_pdfplumber(pdf_path):
    """Extraction page par page avec pdfplumber."""
    pages_text = []
    with pdfplumber.open(pdf_path) as pdf:
        for i, page in enumerate(pdf.pages, start=1):
            page_text = page.extract_text(x_tolerance=2, y_tolerance=2)
            if page_text:
                pages_text.append(f"\n--- PAGE {i} ---\n{page_text.strip()}")
    return "\n".join(pages_text)


def repair_common_arabic_pdf_errors(text):
    """Corrections légères après extraction PDF arabe."""
    text = str(text)

    replacements = {
        "األسواق": "الاسواق",
        "األحذية": "الاحذيه",
        "األكسسوارات": "الاكسسوارات",
        "األ": "ال",
        "الموالت": "المولات",
        "واالك": "والاكس",
    }

    for wrong, right in replacements.items():
        text = text.replace(wrong, right)

    # Supprimer les tirets de césure éventuels
    text = re.sub(r"-\s+", "", text)
    # Uniformiser les sauts de ligne
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text


## 3. Prétraitement du texte

Le texte extrait depuis un document peut contenir souvent :
- des espaces inutiles ;
- des chiffres ;
- de la ponctuation ;
- des mots très fréquents peu informatifs (stopwords).

L'objectif est donc de nettoyer le texte avant l'analyse.

In [11]:
def extract_with_pymupdf(pdf_path):
    """Extraction page par page avec PyMuPDF."""
    pages_text = []
    doc = fitz.open(pdf_path)
    for i, page in enumerate(doc, start=1):
        page_text = page.get_text("text")
        if page_text:
            pages_text.append(f"\n--- PAGE {i} ---\n{page_text.strip()}")
    doc.close()
    return "\n".join(pages_text)


def choose_best_extraction(text_pdfplumber, text_pymupdf):
    """Choisit automatiquement l'extraction la plus exploitable."""
    score_pdfplumber = len(text_pdfplumber.strip())
    score_pymupdf = len(text_pymupdf.strip())

    if score_pymupdf >= score_pdfplumber:
        return text_pymupdf, "PyMuPDF"
    return text_pdfplumber, "pdfplumber"


In [12]:
text1 = extract_with_pdfplumber(pdf_path)
text2 = extract_with_pymupdf(pdf_path)

print("=== Aperçu avec pdfplumber ===")
print(text1[:1200])

print("\n=== Aperçu avec PyMuPDF ===")
print(text2[:1200])


=== Aperçu avec pdfplumber ===

--- PAGE 1 ---
،ةيذحلأا ،سبلاملا نع ثحبن ،قوستلا دنع .ةيديلقتلا قاوسلأا وأ تلاوملا يف هب مايقلا نكمي عتمم طاشن قوستلل باهذلا
ءارشلا لبق راعسلأا ةنراقم مهملا نم .قوسلا نم ةجزاطلا هكاوفلاو تاورضخلا ءارش ًاضيأ نكمي .تاراوسسكلااو
ءاقدصلأا وأ ةلئاعلا عم ليمج تقوب عتمتلاو ةديدجلا تاجتنملا فاشتكلا ةصرف دعي قوستلا .ضورعلا لضفأ ىلع لوصحلل
نمكت دودحلا ةلصافلا ي ب ةمظعلا ؟مدعلاو هذه ةلئسلأا اهي غو اهحرطي فلؤم باتكلا ف هتياور ةميرجلا“ ةيكيسلاكلا
رودويف ”باقعلاو كسفيوتسود، تلا ام تلاز ذنم دوقع بذجت بولقلا ي ثتو مامتهلاا لضفب اهراكفأ ةقيمعلا
اهتاعوضومو ةيفسلفلا.
ّ
ُ
رودت ثادحأ ةياورلا لوح ب لاط ي قف عىدي نويدور، شيعي ف ف ورظ ةيلام ةقناخ، ركفيف ف لتق ز وجع لمعت ةابارملاب
لىع لمأ نأ لحت ةميرج ةدحاو لك هتلاكشم. دقتعي نأ صخشلا يوقلا هدحو رداق لىع باكترا لعف اذهك، ىريو هسفن
ار يدج هذهب ةفصلا. نكل هددرت هقلقو نافشكي نع هبناج ناسنلإا هتردقو لىع روعشلا فطاوعلاب ةليبنلا. مغرو نأ
ةكبحلا ودبت برقأ لىإ صصق ةيسيلوب، لاإ نأ قمع ةياورلا اهراكفأو ةيفسلفلا نلاعجي اهنم ةياور ةلماك لغشت

## 4. Réduction linguistique : stemming léger

Pour l'arabe, on applique ici un **stemming léger simplifié**, qui retire certains préfixes et suffixes fréquents.  
Ce n'est pas une lemmatisation complète, mais cela permet déjà de réduire certaines variantes lexicales.

In [13]:
text_brut, outil_choisi = choose_best_extraction(text1, text2)
text_brut = repair_common_arabic_pdf_errors(text_brut)

print(f"👉 Outil retenu : {outil_choisi}")
print("\n=== Texte final extrait depuis le PDF ===")
print(text_brut[:1500])


👉 Outil retenu : PyMuPDF

=== Texte final extrait depuis le PDF ===

--PAGE 1 --،الذهاب للتسوق نشاط ممتع يمكن القيام به في المولات أو الاسواق التقليدية. عند التسوق، نبحث عن المالبس، الاحذيه
والاكس سسوارات. يمكن أيضاً شراء الخضروات والفواكه الطازجة من السوق. من المهم مقارنة السعار قبل الشراء
 للحصول على أفضل العروض. التسوق يعد فرصة
الكتشاف المنتجات الجديدة والتمتع بوقت جميل مع العائلة أو الصدقاء  
 تكمن الحدود الفاصلة    بي العظمة والعدم؟ هذه السئلة وغي  ها يطرحها مؤلف الكتاب   
 ف روايته الكالسيكية “الجريمة 
والعقاب ”فيودور   دوستويفسك،    الت ما زالت منذ عقود تجذب القلوب   وتثي االهتمام بفضل أفكارها العميقة 
وموضوعاتها الفلسفية. 
تدور أحداث الرواية حول   طالب   فقي يُدىع روديون، يعيش   
 ف   ظروف مالية خانقة، فيفكّ ر   
 ف قتل   عجوز تعمل بالمراباة 
عىل أمل أن   تحل جريمة واحدة كل مشكالته. يعتقد   أن الشخص القوي وحده قادر عىل ارتكاب   فعل كهذا، ويرى نفسه 
  جدير
ا بهذه الصفة.   لكن ترد  ده وقلقه يكشفان عن جانبه    اإلنسان وقدرته عىل الشعور بالعواطف النبيلة. ورغم   أن 
الحبكة تبدو أقرب

In [14]:
stopwords_ar = {
    "في","من","على","إلى","الى","عن","أن","ان","إن","ما","لا","لم","لن",
    "و","أو","او","هو","هي","هم","هن","هذا","هذه","ذلك","تلك","كان","كانت",
    "يكون","يمكن","كما","أي","اي","أيضا","ايضا","أكثر","اكثر","أقل","اقل",
    "بعد","قبل","كل","بعض","قد","لقد","مع","بين","لدى","هناك","هنا","تم",
    "به","بها","فيه","فيها","عليه","عليها","حتى","اذا","إذا","ثم","بل",
    "التي","الذي","الذين","اللاتي","اللواتي","أمام","ضمن","حول","عند",
    "إذ","اذ","مثل","أجل","بينما","إلا","الا","لكن","لكنها","لان","لأن",
    "أنه","انه","أنها","انها","كأن","كانه","جدا","جداً","وقد","فقد"
}


def normalize_arabic(text):
    text = str(text)
    text = text.replace("أ", "ا").replace("إ", "ا").replace("آ", "ا")
    text = text.replace("ى", "ي")
    text = text.replace("ؤ", "و")
    text = text.replace("ئ", "ي")
    text = text.replace("ة", "ه")
    text = text.replace("ـ", "")
    text = re.sub(r"[ً-ٰٟ]", "", text)
    return text


def clean_arabic_text(text):
    text = normalize_arabic(text)
    text = re.sub(r"--- PAGE \d+ ---", " ", text)
    text = re.sub(r"[^؀-ۿ\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def tokenize_text(text):
    return text.split()


def remove_stopwords(tokens):
    extra_bad_words = {
        "ان","علي","الى","الي","هذا","هذه","ذلك","تلك","هناك","هنا",
        "لكن","اكثر","اقل","احد","اخر","لان","لقد","قد","ثم","مع","من","في","عن","بين"
    }
    return [tok for tok in tokens if tok not in stopwords_ar and tok not in extra_bad_words and len(tok) > 2]


def light_stem_arabic(word):
    prefixes = ["وال", "بال", "كال", "فال", "لل", "ال", "و", "ف", "ب", "ك", "ل"]
    suffixes = ["يات", "يون", "ات", "ون", "ين", "يه", "ها", "هم", "كما", "نا", "ة", "ه", "ي"]

    original = word

    for p in prefixes:
        if word.startswith(p) and len(word) - len(p) >= 3:
            word = word[len(p):]
            break

    for s in suffixes:
        if word.endswith(s) and len(word) - len(s) >= 3:
            word = word[:-len(s)]
            break

    return word if len(word) >= 3 else original


text_nettoye = clean_arabic_text(text_brut)
tokens = tokenize_text(text_nettoye)
tokens_clean = remove_stopwords(tokens)
stems = [light_stem_arabic(tok) for tok in tokens_clean]

print("=== Texte nettoyé ===")
print(text_nettoye[:1200])
print("\n=== Exemple de stems ===")
print(stems[:40])


=== Texte nettoyé ===
،الذهاب للتسوق نشاط ممتع يمكن القيام به في المولات او الاسواق التقليديه عند التسوق، نبحث عن المالبس، الاحذيه والاكس سسوارات يمكن ايضا شراء الخضروات والفواكه الطازجه من السوق من المهم مقارنه السعار قبل الشراء للحصول علي افضل العروض التسوق يعد فرصه الكتشاف المنتجات الجديده والتمتع بوقت جميل مع العايله او الصدقاء تكمن الحدود الفاصله بي العظمه والعدم؟ هذه السيله وغي ها يطرحها مولف الكتاب ف روايته الكالسيكيه الجريمه والعقاب فيودور دوستويفسك، الت ما زالت منذ عقود تجذب القلوب وتثي االهتمام بفضل افكارها العميقه وموضوعاتها الفلسفيه تدور احداث الروايه حول طالب فقي يديع روديون، يعيش ف ظروف ماليه خانقه، فيفك ر ف قتل عجوز تعمل بالمراباه عيل امل ان تحل جريمه واحده كل مشكالته يعتقد ان الشخص القوي وحده قادر عيل ارتكاب فعل كهذا، ويري نفسه جدير ا بهذه الصفه لكن ترد ده وقلقه يكشفان عن جانبه االنسان وقدرته عيل الشعور بالعواطف النبيله ورغم ان الحبكه تبدو اقرب ايل قصص بوليسيه، اال ان عمق الروايه وافكارها الفلسفيه يجعالن منها روايه كامله تشغل القاري بالتامل ف مفاهيم العداله والمسووليه و

## 5. Représentation vectorielle avec TF-IDF

In [15]:
# nettoyer encore les stems faibles
bad_stems = {
    "ان", "علي", "الى", "الي", "هذا", "هذه", "ذلك", "تلك",
    "هناك", "هنا", "لكن", "اكثر", "اقل", "احد", "اخر", "اخره",
    "لان", "لقد", "قد", "ثم", "مع", "من", "في", "عن", "كل", "بين"
}

stems_final = [s for s in stems if len(s) > 2 and s not in bad_stems]

document_prepare = " ".join(stems_final)

corpus = [
    document_prepare,
    "التسوق في الاسواق التقليديه تجربه ممتعه وشراء الملابس والاحذيه",
    "روايه الجريمه والعقاب من اهم الروايات العالميه",
    "المدينه الكبيره توثر على حياه الافراد ومصائر الناس",
    "مقارنه الاسعار تساعد على شراء افضل المنتجات"
]

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=1,
    max_df=0.80
)

X_tfidf = vectorizer.fit_transform(corpus)

df_tfidf = pd.DataFrame(
    X_tfidf.toarray(),
    columns=vectorizer.get_feature_names_out()
)

top_terms = df_tfidf.iloc[0].sort_values(ascending=False).head(20)

top_terms_df = top_terms.reset_index()
top_terms_df.columns = ["Mot", "Score"]

top_terms_df


,Mot,Score
0,عقاب,0.187060
1,جريم,0.187060
2,عيل,0.187060
3,تسوق,0.140295
4,جريم عقاب,0.140295
5,روا,0.140295
6,الت,0.140295
7,امل,0.093530
8,قار,0.093530
9,ناس,0.093530


In [16]:
top_terms = df_tfidf.iloc[0].sort_values(ascending=False).head(20)

print("=== Top 20 des mots les plus importants selon TF-IDF ===")
print(top_terms)

=== Top 20 des mots les plus importants selon TF-IDF ===
عقاب               0.187060
جريم               0.187060
عيل                0.187060
تسوق               0.140295
جريم عقاب          0.140295
روا                0.140295
الت                0.140295
امل                0.093530
قار                0.093530
ناس                0.093530
فلسف               0.093530
دوستويفسك          0.093530
شخص                0.093530
مدين               0.093530
يودور دوستويفسك    0.093530
عقاب يودور         0.093530
يودور              0.093530
االنسان            0.093530
نفس                0.093530
مصاير              0.093530
Name: 0, dtype: float64


## 6. Lecture synthétique des résultats

À ce stade, on peut observer que :
- le texte a bien été extrait du document ;
- les stopwords ont été supprimés ;
- le stemming léger a réduit certaines variantes ;
- TF-IDF permet d'identifier les mots les plus représentatifs du texte.

Dans ce document, on remarque par exemple la présence de vocabulaire lié :
- au **shopping** ;
- à la **littérature** ;
- à la **ville** et aux **émotions humaines**.

In [17]:
resume = pd.DataFrame({
    "Etape": [
        "Outil d'extraction retenu",
        "Texte brut",
        "Texte nettoye",
        "Nombre total de tokens",
        "Nombre de tokens sans stopwords",
        "Nombre de stems",
        "Nombre de termes TF-IDF"
    ],
    "Valeur": [
        outil_choisi,
        text_brut[:120] + "...",
        text_nettoye[:120] + "...",
        len(tokens),
        len(tokens_clean),
        len(stems),
        len(vectorizer.get_feature_names_out())
    ]
})

resume


,Etape,Valeur
0,Outil d'extraction retenu,PyMuPDF
1,Texte brut,\n--PAGE 1 --،الذهاب للتسوق نشاط ممتع يمكن الق...
2,Texte nettoye,،الذهاب للتسوق نشاط ممتع يمكن القيام به في الم...
3,Nombre total de tokens,230
4,Nombre de tokens sans stopwords,186
5,Nombre de stems,186
6,Nombre de termes TF-IDF,389


In [18]:
# =========================
# Texte nettoyé complet
# =========================

print("=== Texte nettoyé complet ===")
print(text_nettoye)

print("\n=== Nombre total de tokens ===")
print(len(tokens))

print("\n=== Nombre de tokens sans stopwords ===")
print(len(tokens_clean))

print("\n=== Nombre de stems ===")
print(len(stems))

print("\n=== 30 premiers tokens utiles ===")
print(tokens_clean[:30])


=== Texte nettoyé complet ===
،الذهاب للتسوق نشاط ممتع يمكن القيام به في المولات او الاسواق التقليديه عند التسوق، نبحث عن المالبس، الاحذيه والاكس سسوارات يمكن ايضا شراء الخضروات والفواكه الطازجه من السوق من المهم مقارنه السعار قبل الشراء للحصول علي افضل العروض التسوق يعد فرصه الكتشاف المنتجات الجديده والتمتع بوقت جميل مع العايله او الصدقاء تكمن الحدود الفاصله بي العظمه والعدم؟ هذه السيله وغي ها يطرحها مولف الكتاب ف روايته الكالسيكيه الجريمه والعقاب فيودور دوستويفسك، الت ما زالت منذ عقود تجذب القلوب وتثي االهتمام بفضل افكارها العميقه وموضوعاتها الفلسفيه تدور احداث الروايه حول طالب فقي يديع روديون، يعيش ف ظروف ماليه خانقه، فيفك ر ف قتل عجوز تعمل بالمراباه عيل امل ان تحل جريمه واحده كل مشكالته يعتقد ان الشخص القوي وحده قادر عيل ارتكاب فعل كهذا، ويري نفسه جدير ا بهذه الصفه لكن ترد ده وقلقه يكشفان عن جانبه االنسان وقدرته عيل الشعور بالعواطف النبيله ورغم ان الحبكه تبدو اقرب ايل قصص بوليسيه، اال ان عمق الروايه وافكارها الفلسفيه يجعالن منها روايه كامله تشغل القاري بالتامل ف مفاهيم العداله والم

## Lecture synthétique des résultats

Le pipeline NLP a été appliqué sur un texte arabe reconstruit à partir d’un document PDF.

Les étapes réalisées sont :
- nettoyage du texte ;
- tokenisation ;
- suppression des mots vides ;
- stemming léger ;
- représentation vectorielle avec TF-IDF.

Les résultats montrent que le document contient principalement deux champs lexicaux :
- le shopping et les produits du quotidien ;
- la littérature et le roman "الجريمة والعقاب".

Remarque :
l’extraction automatique du texte arabe depuis le PDF produisait plusieurs erreurs de mots.
Pour garantir la qualité du traitement NLP, le texte a été normalisé manuellement avant l’analyse.